# Day 40 — statistics (awaiting cleaned CSV)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
f=Path('../Customer_Churn_Prediction/data/processed/cleaned_customer_churn.csv')
assert f.exists(), 'Run Day 38 first'
df=pd.read_csv(f); df['ChurnFlag']=df.Churn.map({'Yes':1,'No':0}); assert df.ChurnFlag.notna().all()
import seaborn as sns
from scipy import stats
num=df.select_dtypes('number').drop(columns=['CustomerID'],errors='ignore'); display(num.describe().T)
for c in num.columns.drop('ChurnFlag',errors='ignore'): display(df.groupby('Churn')[c].agg(['count','mean','median','std']))
corr=num.corr(); display(corr); plt.figure(figsize=(13,10)); sns.heatmap(corr,cmap='vlag',center=0); plt.title('Numerical correlation'); plt.tight_layout(); plt.savefig('Correlation_Heatmap.png'); plt.show()
print('Positive: move together; negative: opposite; near zero: weak linear association. Correlation is not causation; categorical associations require separate tests.')
alpha=.05
if 'ContractType' in df:
    table=pd.crosstab(df.ContractType,df.Churn); display(table)
    if min(table.shape)>=2:
        stat,p,dof,expected=stats.chi2_contingency(table); print('Research question: contract and churn association. H0: independent; H1: associated. Expected counts:',expected)
        if (expected>=5).all(): print('Chi-square',stat,'p=',p,'Reject H0' if p<alpha else 'Fail to reject H0')
        else: print('Expected counts too small; do not interpret asymptotic p-value')
if 'MonthlyCharges' in df:
    a=df.loc[df.Churn=='Yes','MonthlyCharges'].dropna(); b=df.loc[df.Churn=='No','MonthlyCharges'].dropna(); print('Group sizes',len(a),len(b)); display(a.describe(),b.describe())
    if min(len(a),len(b))>=30:
        print('Welch t-test: H0 equal means; H1 different means. Verify independent customers and distributions before interpretation.')
        t,p=stats.ttest_ind(a,b,equal_var=False); print('t=',t,'p=',p,'Reject H0' if p<alpha else 'Fail to reject H0','Mean difference',a.mean()-b.mean())
print('Alpha=.05; p-value is probability of a result at least as extreme under H0 and assumptions. Significance is not causation or practical importance. Confirm independence and account for multiple tests.')
